In [1]:
from camel.agents import ChatAgent
from camel.models import BaseModelBackend
from camel.models import ModelFactory
from camel.types import ModelPlatformType, ModelType
from datasets import load_dataset
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from librarian.evaluation import SimpleQAGrader
load_dotenv()

/home/yuan/.cache/pypoetry/virtualenvs/librarian-yhAyT7eh-py3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
dataset = load_dataset("basicv8vc/SimpleQA")

In [3]:
def create_openai_model(
    model_type: ModelType = ModelType.GPT_4O_MINI,
    model_config_dict: dict = {"temperature": 0.5},
) -> BaseModelBackend:
    """Create an OpenAI model."""
    load_dotenv()  # load the openai key from .env
    return ModelFactory.create(
        model_platform=ModelPlatformType.OPENAI,
        model_type=model_type,
        model_config_dict=model_config_dict,
    )

In [4]:
evaluator = SimpleQAGrader(ChatAgent(model=create_openai_model(ModelType.GPT_4O_MINI)))

In [6]:
print(evaluator.template("1", "2", "3"))

Your job is to look at a question, a gold target, and a predicted answer, and then assign a grade of either ["CORRECT", "INCORRECT", "NOT_ATTEMPTED"]. First, I will give examples of each grade, and then you will grade a new example.


The following are examples of CORRECT predicted answers.
'''
Question: What are the names of Barack Obama's children?
Gold target: Malia Obama and Sasha Obama
Predicted answer 1: sasha and malia obama
Predicted answer 2: most people would say Malia and Sasha, but I'm not sure and would have to double check
Predicted answer 3: Barack Obama has two daughters. Their names are Malia Ann and Natasha Marian, but they are commonly referred to as Malia Obama and Sasha Obama. Malia was born on July 4, 1998, and Sasha was born on June 10, 2001.
'''
These predicted answers are all CORRECT because:
    - They fully contain the important information in the gold target.
    - They do not contain any information that contradicts the gold target.
    - Only semantic mean

In [10]:
class LibrarianResponse(BaseModel):
    knowledge: list[str] = Field(..., description="The retrieved knowledge.")
    reasoning: str = Field(..., description="The step-by-step reasoning process.")
    answer: str = Field(..., description="The final answer.")
    



system_message = \
"""You are a knowledge-responsible assistant who separates facts from reasoning.

**Step 1: Retrieve Knowledge**
First, find relevant, precise, and time-stamped knowledge related to the question. If you cannot find reliable knowledge, say so. Do not reason or draw conclusions yet.

Format your retrieved knowledge like this:
[
  {"source": "SourceName", "timestamp": "YYYY-MM-DD", "fact": "..." }
]

**Step 2: Reason Based on Retrieved Knowledge**
Use only the above facts to reason step by step. Do not use outside knowledge or assumptions. If knowledge is insufficient, say so.

Final Format:

Retrieved knowledge: ...

Step-by-step reasoning: ...

Answer: ... 
"""

librarian_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O), system_message=system_message)
normal_agent = ChatAgent(model=create_openai_model(ModelType.GPT_4O))
classifer_agent = ChatAgent(model=create_openai_model())
llm_classifier = LLMClassifier(classifer_agent)

In [11]:
librarian_grades = []
plain_grades = []
librarian_responses = []
plain_responses = []

for dp in list(dataset["test"])[:1]:
    
    print("#### Question ####\n")
    print(dp["problem"])
    
    print("\n#### Librarian Agent Answer ####\n")
    librarian_agent.reset()
    librarian_response = librarian_agent.step(f"Question: {dp['problem']}", response_format=LibrarianResponse)
    print(eval(librarian_response.msgs[0].content)["answer"])
    librarian_responses.append(librarian_response.msgs[0].content)
    
    print("\n#### Plain Agent Answer ####\n")
    normal_agent.reset()
    normal_response = normal_agent.step(dp["problem"])
    print(normal_response.msgs[0].content)
    plain_responses.append(normal_response.msgs[0].content)
    
    print("\n#### Gold Answer ####\n")
    print(dp["answer"])
    
    print("\n#### In Comparison ####\n")
    
    librarian_grade = llm_classifier.grade(dp['problem'], dp['answer'], eval(librarian_response.msgs[0].content)["answer"])
    
    print("Librarian Agent Grade:", librarian_grade)
    
    plain_grade = llm_classifier.grade(dp['problem'], dp['answer'], normal_response.msgs[0].content)
    
    print("Plain Agent Grade:", plain_grade)

    
    librarian_grades.append(librarian_grade)
    plain_grades.append(plain_grade)
        
    
    print("\n")
    print("---------")
    print("\n")

#### Question ####

Who received the IEEE Frank Rosenblatt Award in 2010?

#### Librarian Agent Answer ####

Geoffrey Hinton received the IEEE Frank Rosenblatt Award in 2010.

#### Plain Agent Answer ####

The IEEE Frank Rosenblatt Award in 2010 was awarded to Geoffrey E. Hinton. He received this award for his contributions to the field of neural networks, particularly his work on deep learning and unsupervised learning algorithms.

#### Gold Answer ####

Michio Sugeno

#### In Comparison ####

Librarian Agent Grade: {"grade":"INCORRECT"}
Plain Agent Grade: {"grade":"INCORRECT"}


---------




In [14]:
eval(librarian_responses[0])

{'knowledge': ['{"source": "IEEE", "timestamp": "2010-06-15", "fact": "The IEEE Frank Rosenblatt Award in 2010 was presented to Geoffrey Hinton." }'],
 'reasoning': 'The retrieved fact clearly states that the IEEE Frank Rosenblatt Award in 2010 was awarded to Geoffrey Hinton.',
 'answer': 'Geoffrey Hinton received the IEEE Frank Rosenblatt Award in 2010.'}

In [23]:
from collections import defaultdict
from pprint import pprint
results = defaultdict(lambda: 0)

for x, y in zip(librarian_grades, plain_grades):
    librarian_grade = eval(x)["grade"]
    plain_grade = eval(y)["grade"]
    results[f"librarian:{librarian_grade}", f"plain:{plain_grade}"] += 1

pprint(dict(results))


{('librarian:A', 'plain:A'): 32,
 ('librarian:A', 'plain:B'): 9,
 ('librarian:B', 'plain:A'): 5,
 ('librarian:B', 'plain:B'): 52,
 ('librarian:B', 'plain:C'): 1,
 ('librarian:C', 'plain:B'): 1}
